# Chopp & Cia · Notebook 01 — Ingestão do consolidado

## Responsabilidade única

Receber o **CSV consolidado**, validar seu contrato e publicar uma tabela Delta
versionada no Unity Catalog. O dump SQL e todo o parser do ERP deixam de fazer parte
deste pipeline.

| Ambiente | Entrada | Saída |
| :--- | :--- | :--- |
| Databricks | caminho `/Volumes/.../dataset.csv` | `dataset_consolidado_v<versão>` |
| Windows / Jupyter | seletor de arquivo | validação local do CSV |

## 1. Ambiente e parâmetros

No Databricks, execute a célula uma vez, preencha o caminho do CSV no campo criado
no topo e execute novamente. No Windows, o seletor abre diretamente.

In [ ]:
# AMBIENTE E PARÂMETROS
from pathlib import Path
from datetime import datetime
import hashlib
import re

import numpy as np
import pandas as pd

try:
    dbutils  # type: ignore[name-defined]
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False


def widget_texto(nome: str, padrao: str, rotulo: str) -> str:
    """Cria o campo no Databricks uma vez e preserva o valor digitado."""
    if not EM_DATABRICKS:
        return padrao
    try:
        return dbutils.widgets.get(nome)  # type: ignore[name-defined]
    except Exception:
        dbutils.widgets.text(nome, padrao, rotulo)  # type: ignore[name-defined]
        return dbutils.widgets.get(nome)  # type: ignore[name-defined]


def selecionar_csv_local() -> str:
    """Abre o seletor do Windows para o CSV consolidado."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.update()
    try:
        caminho = filedialog.askopenfilename(
            parent=root,
            title="Selecione o dataset consolidado",
            filetypes=[("CSV", "*.csv"), ("Todos os arquivos", "*.*")],
        )
    finally:
        root.destroy()
    if not caminho:
        raise ValueError("Nenhum CSV foi selecionado.")
    return caminho


if EM_DATABRICKS:
    CSV_CONSOLIDADO = widget_texto(
        "csv_consolidado", "", "Caminho do CSV consolidado no Volume"
    ).strip()
    CATALOGO = widget_texto("catalogo", "projetointegrador", "Catálogo")
    SCHEMA = widget_texto("schema", "projetointegrador", "Schema")
    DATA_VERSION = widget_texto("data_version", "1.0", "Versão dos dados")
else:
    CSV_CONSOLIDADO = selecionar_csv_local()
    CATALOGO = "projetointegrador"
    SCHEMA = "projetointegrador"
    DATA_VERSION = "1.0"

if EM_DATABRICKS and not CSV_CONSOLIDADO:
    raise ValueError(
        "Preencha o campo 'Caminho do CSV consolidado no Volume' no topo "
        "e execute esta célula novamente."
    )
if not re.fullmatch(r"\d+\.\d+", DATA_VERSION):
    raise ValueError("DATA_VERSION deve seguir o formato MAIOR.MENOR, por exemplo 1.0.")

CAMINHO_CSV = Path(CSV_CONSOLIDADO)
TABELA_DESTINO = f"{CATALOGO}.{SCHEMA}.dataset_consolidado_v{DATA_VERSION.replace('.', '_')}"
VIEW_CORRENTE = f"{CATALOGO}.{SCHEMA}.dataset_consolidado_corrente"
TABELA_CATALOGO_VERSOES = f"{CATALOGO}.{SCHEMA}.catalogo_versoes_dados"
PERMITIR_SOBRESCRITA = False

print(f"Ambiente : {'Databricks' if EM_DATABRICKS else 'Local'}")
print(f"Entrada  : {CAMINHO_CSV}")
print(f"Destino  : {TABELA_DESTINO}")

## 2. Carga e contrato

In [ ]:
# CARGA E CONTRATO DO CSV CONSOLIDADO
if not CAMINHO_CSV.exists():
    raise FileNotFoundError(f"CSV não encontrado: {CAMINHO_CSV}")

try:
    dataset = pd.read_csv(CAMINHO_CSV, sep=";", encoding="utf-8-sig", low_memory=False)
except UnicodeDecodeError:
    dataset = pd.read_csv(CAMINHO_CSV, sep=";", encoding="latin-1", low_memory=False)

dataset.columns = [
    str(c).strip().upper().replace(" ", "_").replace("-", "_").replace(".", "_")
    for c in dataset.columns
]

COLUNAS_CONTRATO = [
    "ID_PESSOA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO",
    "PRIMEIRA_COMPRA", "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO",
    "TICKET_MEDIO", "PRODUTO_FAVORITO", "TOTAL_PARCELAS", "PARCELAS_ATRASADAS",
    "TAXA_ATRASO_PAGAMENTO", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "VALOR_TOTAL_PARCELAS", "AGING_PAGAMENTO", "TOTAL_COMODATOS",
    "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO", "MEDIA_DIAS_ATRASO_COM",
    "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS", "AGING_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO", "CORE_BUSINESS",
    "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]

faltantes = sorted(set(COLUNAS_CONTRATO) - set(dataset.columns))
if faltantes:
    raise ValueError(f"CSV fora do contrato. Colunas ausentes: {faltantes}")

if dataset.empty:
    raise ValueError("O CSV consolidado está vazio.")
if dataset["ID_PESSOA"].isna().any():
    raise ValueError("ID_PESSOA contém valores vazios.")
if dataset["ID_PESSOA"].duplicated().any():
    raise ValueError("ID_PESSOA duplicado: o consolidado deve ter uma linha por cliente.")

proibidas = ("NOME", "NM_", "FANTASIA", "RAZAO", "CPF", "CNPJ", "EMAIL", "TELEFONE", "ENDERECO")
identificadoras = [c for c in dataset.columns if any(p in c.upper() for p in proibidas)]
if identificadoras:
    raise ValueError(f"Colunas de identificação nominal não permitidas: {identificadoras}")

for coluna in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    dataset[coluna] = pd.to_datetime(dataset[coluna], errors="coerce")

numericas = [
    c for c in COLUNAS_CONTRATO
    if c not in {
        "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO", "PRIMEIRA_COMPRA",
        "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA", "PRODUTO_FAVORITO",
        "AGING_PAGAMENTO", "AGING_COMODATO", "PERFIL_RISCO",
    }
]
for coluna in numericas:
    dataset[coluna] = pd.to_numeric(dataset[coluna], errors="coerce")

sha256 = hashlib.sha256()
with CAMINHO_CSV.open("rb") as arquivo:
    for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
        sha256.update(bloco)
INGESTAO_HASH = sha256.hexdigest()

dataset["_DATA_VERSION"] = DATA_VERSION
dataset["_INGESTAO_HASH"] = INGESTAO_HASH
dataset["_CARGA_TS"] = datetime.now().isoformat(timespec="seconds")

print(f"CSV validado: {dataset.shape[0]:,} clientes × {dataset.shape[1]} colunas")
print(f"SHA-256     : {INGESTAO_HASH}")

## 3. Auditoria

In [ ]:
# AUDITORIA RÁPIDA
print("=" * 72)
print("AUDITORIA DO CONSOLIDADO")
print("=" * 72)
print(f"Clientes únicos       : {dataset['ID_PESSOA'].nunique():,}")
print(f"Clientes core business: {int(dataset['CORE_BUSINESS'].fillna(0).sum()):,}")
print(f"Com vendas            : {int(dataset['TEM_VENDAS'].fillna(0).sum()):,}")
print(f"Com financeiro        : {int(dataset['TEM_FINANCEIRO'].fillna(0).sum()):,}")
print(f"Com comodato          : {int(dataset['TEM_COMODATO'].fillna(0).sum()):,}")
print(f"Duplicidade de ID     : {int(dataset['ID_PESSOA'].duplicated().sum())}")
print("=" * 72)

## 4. Publicação no Unity Catalog

In [ ]:
# PUBLICAÇÃO DA TABELA VERSIONADA
if not EM_DATABRICKS:
    print("Execução local concluída: CSV validado; nenhuma tabela foi publicada.")
else:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}")  # type: ignore[name-defined]

    if spark.catalog.tableExists(TABELA_DESTINO) and not PERMITIR_SOBRESCRITA:  # type: ignore[name-defined]
        raise ValueError(
            f"A tabela {TABELA_DESTINO} já existe. Incremente DATA_VERSION ou defina "
            "PERMITIR_SOBRESCRITA = True conscientemente."
        )

    pronto = dataset.copy()
    for coluna in pronto.select_dtypes(include="object").columns:
        pronto[coluna] = pronto[coluna].fillna("").astype(str)

    spark_df = spark.createDataFrame(pronto)  # type: ignore[name-defined]
    modo = "overwrite" if PERMITIR_SOBRESCRITA else "errorifexists"
    (
        spark_df.write.format("delta")
        .mode(modo)
        .option("overwriteSchema", "true")
        .saveAsTable(TABELA_DESTINO)
    )

    spark.sql(f"CREATE OR REPLACE VIEW {VIEW_CORRENTE} AS SELECT * FROM {TABELA_DESTINO}")  # type: ignore[name-defined]

    origem = CAMINHO_CSV.name.replace("'", "''")
    spark.sql(f"""  # type: ignore[name-defined]
        ALTER TABLE {TABELA_DESTINO} SET TBLPROPERTIES (
            'data_version' = '{DATA_VERSION}',
            'ingestao_hash' = '{INGESTAO_HASH}',
            'origem_csv' = '{origem}'
        )
    """)

    registro = spark.createDataFrame([{  # type: ignore[name-defined]
        "data_version": DATA_VERSION,
        "tabela": TABELA_DESTINO,
        "ingestao_hash": INGESTAO_HASH,
        "origem_csv": CAMINHO_CSV.name,
        "carga_ts": datetime.now(),
        "n_clientes": int(len(dataset)),
    }])
    modo_catalogo = "append" if spark.catalog.tableExists(TABELA_CATALOGO_VERSOES) else "overwrite"  # type: ignore[name-defined]
    registro.write.format("delta").mode(modo_catalogo).saveAsTable(TABELA_CATALOGO_VERSOES)

    print(f"Tabela publicada: {TABELA_DESTINO}")
    print(f"View corrente   : {VIEW_CORRENTE}")